<a href="https://colab.research.google.com/github/korzhimanov/dsp-seminars/blob/main/seminars/8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Практическое занятие №8: Вейвлет-преобразования

## Подготовка окружения

In [ ]:
!pip install -q PyWavelets

import numpy as np
import matplotlib.pyplot as plt
import pywt
from scipy import signal
from scipy.fft import dct, idct

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 4)

## Часть 1. Непрерывное вейвлет-преобразование

### Задание 1.1. Масштаб и частота

1. Выведите центральные частоты (`pywt.central_frequency`) вейвлетов `'mexh'`, `'morl'`, `'cmor1.5-1.0'`, `'gaus1'`.
2. Сформируйте тон 50 Гц длительностью 1 с при $f_s = 1000$ Гц. Задайте 100 частот анализа в геометрической прогрессии от 10 до 200 Гц (`np.geomspace`) и пересчитайте их в масштабы по формуле $a = f_c f_s / f$.
3. Вычислите CWT с вейвлетом `'cmor1.5-1.0'` (`pywt.cwt(x, scales, wavelet, sampling_period=1/fs)`). Постройте зависимость $|W|$, усреднённого по центральной половине записи, от частоты. Совпадает ли максимум с 50 Гц?
4. Повторите для вейвлетов `'mexh'` и `'gaus1'`. Постройте для них также зависимость $|W(a, b)|$ от времени $b$ на масштабе, соответствующем 50 Гц.

In [ ]:
fs = 1000
t = np.arange(0, 1, 1/fs)
x = np.sin(2*np.pi*50*t)
# Ваш код здесь

**Вопросы:**
- Насколько точно максимум $|W|$ указывает частоту тона для каждого вейвлета?
- Почему для вещественных вейвлетов $|W(a,b)|$ осциллирует во времени, а для комплексного вейвлета Морле – нет?
- Почему для некоторых вейвлетов максимум заметно смещён относительно 50 Гц? (Подсказка: центральная частота – лишь приближённая характеристика широкополосного вейвлета; сравните спектры вейвлетов.)

{Введите ваш ответ здесь}

### Задание 1.2. Скалограмма нестационарного сигнала

1. Сформируйте сигнал длительностью 2 с при $f_s = 1000$ Гц: тон 20 Гц амплитудой 0.6 на всей длительности; чирп от 40 до 250 Гц на интервале 0.4–1.6 с (`signal.chirp`); два щелчка (гауссовы импульсы шириной 2 мс) в моменты 0.25 и 1.8 с.
2. Постройте скалограмму $|W(a,b)|$ (вейвлет `'cmor1.5-1.0'`, 150 частот от 4 до 400 Гц) с логарифмической осью частот (`plt.pcolormesh` и `plt.yscale('log')`).
3. Нанесите на скалограмму **конус влияния**: для вейвлета `cmorB-C` на масштабе $a$ краевые эффекты распространяются на расстояние $a\sqrt{B/2}$ отсчётов от краёв записи.
4. Постройте спектрограмму того же сигнала с окном 128 отсчётов и сравните её со скалограммой.

In [ ]:
fs = 1000
t = np.arange(0, 2, 1/fs)
# Ваш код здесь

**Вопросы:**
- Как выглядят на скалограмме тон, чирп и щелчки? Почему щелчки сужаются к высоким частотам?
- Какие детали лучше видны на скалограмме, а какие – на спектрограмме?

{Введите ваш ответ здесь}

### Задание 1.3. Параметр ширины вейвлета Морле

1. Сформируйте сигнал длительностью 2 с при $f_s = 1000$ Гц: сумма тонов 40 и 50 Гц на интервале 0–1 с и два щелчка (одиночные отсчёты амплитуды 20) в моменты 1.50 и 1.52 с.
2. Постройте скалограммы с вейвлетами `'cmorB-1.0'` при $B = 0.5,\ 1.5,\ 5,\ 20$ (частоты 20–150 Гц).
3. Для каждого $B$ определите, различимы ли тоны 40 и 50 Гц и различимы ли два щелчка на частотах около 100 Гц. Для анализа щелчков постройте увеличенный фрагмент скалограммы на интервале 1.45–1.57 с.

In [ ]:
# Ваш код здесь

**Вопросы:**
- При каких $B$ разрешаются тоны? При каких – щелчки? Существует ли $B$, при котором разрешается и то, и другое?
- Как этот результат связан с принципом неопределённости?

{Введите ваш ответ здесь}

## Часть 2. Дискретное вейвлет-преобразование

### Задание 2.1. Преобразование Хаара вручную

1. Допишите функцию `haar_step(a)`, выполняющую один уровень ортонормированного преобразования Хаара: $s_k = (a_{2k} + a_{2k+1})/\sqrt2$, $\;d_k = (a_{2k} - a_{2k+1})/\sqrt2$.
2. Выполните полное разложение сигнала $x = [4,\,2,\,5,\,5,\,1,\,3,\,6,\,6]$ (повторяя шаг для аппроксимации, пока не останется один коэффициент). Сравните с `pywt.wavedec(x, 'haar')`.
3. Напишите обратное преобразование и убедитесь, что сигнал восстанавливается точно.
4. Проверьте равенство Парсеваля: энергия сигнала равна энергии коэффициентов.

In [ ]:
def haar_step(a):
    """Один уровень преобразования Хаара: возвращает (аппроксимация, детали)."""
    # Ваш код здесь
    pass

def haar_inverse_step(s, d):
    """Обратный шаг: восстанавливает сигнал по аппроксимации и деталям."""
    # Ваш код здесь
    pass

x = np.array([4, 2, 5, 5, 1, 3, 6, 6], dtype=float)
# Ваш код здесь

**Вопрос:** Почему многие коэффициенты оказались нулевыми? Какие свойства сигнала они отражают?

{Введите ваш ответ здесь}

### Задание 2.2. Один уровень алгоритма Малла

1. Возьмите фильтры вейвлета Добеши db4: `w = pywt.Wavelet('db4')`, фильтры анализа `w.dec_lo`, `w.dec_hi`, синтеза – `w.rec_lo`, `w.rec_hi`.
2. Реализуйте один уровень разложения: свёртка с фильтром анализа (`np.convolve`) и прореживание – взять отсчёты с нечётными индексами (`[1::2]`). Сравните результат с `pywt.dwt(x, 'db4', mode='zero')` для случайного сигнала длиной 64.
3. Реализуйте синтез: вставка нулей между коэффициентами, свёртка с фильтрами синтеза, сложение ветвей. Восстановленный сигнал начинается с отсчёта номер $L - 2$, где $L$ – длина фильтра.
4. Постройте АЧХ фильтров `dec_lo` и `dec_hi` и проверьте, что $|H|^2 + |G|^2 = 2$.

In [ ]:
w = pywt.Wavelet('db4')
np.random.seed(0)
x = np.random.randn(64)
# Ваш код здесь

**Вопросы:**
- Почему после прореживания информация не теряется, хотя каждая ветвь содержит вдвое меньше отсчётов?
- Почему сложность полного многоуровневого разложения пропорциональна $N$?

{Введите ваш ответ здесь}

### Задание 2.3. Многоуровневое разложение и частотные полосы

1. Сформируйте сигнал $x(t) = \sin(2\pi\cdot 10t) + \sin(2\pi\cdot 60t) + \sin(2\pi\cdot 200t)$ длительностью 2.048 с при $f_s = 1000$ Гц.
2. Выполните разложение `pywt.wavedec(x, 'db8', level=6)`. Учтите порядок коэффициентов в списке: $[a_6, d_6, d_5, \dots, d_1]$.
3. Составьте таблицу: уровень, полоса частот ($d_j$ соответствует $[f_s/2^{j+1},\, f_s/2^j]$, $a_J$ – $[0,\, f_s/2^{J+1}]$), доля энергии сигнала в коэффициентах уровня.
4. Восстановите вклад каждого уровня в отдельности (обнулив остальные коэффициенты и применив `pywt.waverec`) и постройте графики.

In [ ]:
fs = 1000
t = np.arange(0, 2.048, 1/fs)
# Ваш код здесь

**Вопросы:**
- В коэффициентах каких уровней оказался каждый из тонов?
- Почему энергия тона 60 Гц распределилась между двумя уровнями?

{Введите ваш ответ здесь}

### Задание 2.4. Нулевые моменты

1. Сформируйте кусочно-полиномиальный сигнал из 512 отсчётов на отрезке $[0, 1)$: прямая $2t$ при $t < 0.5$, парабола $1.5 - 4(t - 0.5)^2$ при $t \ge 0.5$ и скачок на $+0.5$ при $t \ge 0.75$.
2. Выполните один уровень DWT (`pywt.dwt(x, wavelet, mode='periodization')`) для вейвлетов `haar`, `db2`, `db3`, `db4`. Постройте детали $d_1$.
3. Для каждого вейвлета подсчитайте число деталей с модулем больше $10^{-6}$. Число нулевых моментов вейвлета можно узнать из `pywt.Wavelet(name).vanishing_moments_psi`.

In [ ]:
N = 512
n = np.arange(N) / N
# Ваш код здесь

**Вопросы:**
- Как число ненулевых деталей связано с числом нулевых моментов вейвлета?
- Почему ненулевые детали появляются у краёв записи?

{Введите ваш ответ здесь}

## Часть 3. Применения

### Задание 3.1. Подавление шума

1. Возьмите тестовые сигналы `pywt.data.demo_signal('Blocks', 1024)` и `pywt.data.demo_signal('Doppler', 1024)`, нормируйте их на стандартное отклонение и добавьте белый шум с σ = 0.25.
2. Реализуйте вейвлет-денойзинг: разложение `wavedec` (5 уровней, `mode='periodization'`), оценка шума $\hat\sigma = \operatorname{median}(|d_1|)/0.6745$, универсальный порог $\lambda = \hat\sigma\sqrt{2\ln N}$, пороговая обработка всех деталей (`pywt.threshold`), обратное преобразование.
3. Сравните мягкий и жёсткий пороги и вейвлеты `haar`, `db4`, `sym8`. Сведите среднеквадратичные ошибки в таблицу.
4. Сравните лучший результат с фильтром Баттерворта 4-го порядка (`signal.butter` + `signal.filtfilt`), подобрав частоту среза так, чтобы ошибка была минимальной.

In [ ]:
np.random.seed(3)
N = 1024
# Ваш код здесь

**Вопросы:**
- Какой вейвлет и какой порог лучше подходят для каждого сигнала? Почему?
- Насколько близка оценка $\hat\sigma$ к истинному уровню шума?
- Почему вейвлет-денойзинг сохраняет фронты лучше линейного фильтра?

{Введите ваш ответ здесь}

### Задание 3.2. Сжатие сигнала

1. Возьмите сигнал `pywt.data.demo_signal('Piece-Regular', 1024)`.
2. Выполните полное разложение (`wavedec`, вейвлет `db4`, `mode='periodization'`) и сохраните долю $p$ наибольших по модулю коэффициентов, обнулив остальные (удобно использовать `pywt.coeffs_to_array` и `pywt.array_to_coeffs`). Восстановите сигнал.
3. Сделайте то же с коэффициентами ДКП (`dct(x, norm='ortho')`, `idct`).
4. Постройте зависимость относительной ошибки $\|\hat x - x\|/\|x\|$ от $p$ в диапазоне 1–20 % для обоих преобразований (логарифмическая шкала по оси ошибки).
5. Повторите для гладкого сигнала `'HeaviSine'`.

In [ ]:
# Ваш код здесь

**Вопросы:**
- Для какого сигнала вейвлет-преобразование выигрывает у ДКП, а для какого – нет? Почему?

{Введите ваш ответ здесь}

### Задание 3.3. Сжатие изображения

1. Загрузите изображение `pywt.data.camera()` (512×512, оттенки серого).
2. Выполните двумерное разложение `pywt.wavedec2(img, wavelet, level=3, mode='periodization')`. Изобразите коэффициенты (`pywt.coeffs_to_array`), нормировав каждую субполосу.
3. Сохраните аппроксимацию целиком, а из деталей – наибольшие по модулю коэффициенты так, чтобы всего осталось 1, 2, 5 и 10 % коэффициентов. Восстановите изображение и вычислите PSNR: $10\lg\dfrac{255^2}{\overline{(\hat I - I)^2}}$.
4. Сравните вейвлеты `haar`, `db4` и `bior4.4` (последний близок к CDF 9/7 из JPEG2000).

In [ ]:
img = pywt.data.camera().astype(float)
# Ваш код здесь

**Вопросы:**
- Какие артефакты появляются при сильном сжатии у разных вейвлетов?
- Какой вейвлет даёт наибольший PSNR? Почему в JPEG2000 используют биортогональные вейвлеты?

{Введите ваш ответ здесь}

### Задание 3.4. Обработка ЭКГ

1. Загрузите запись ЭКГ `pywt.data.ecg()` (частота дискретизации 360 Гц), вычтите среднее и нормируйте на максимум модуля.
2. Добавьте типичные помехи: дрейф изолинии $0.8\sin(2\pi\cdot 0.3t + 1)$, сетевую наводку $0.15\sin(2\pi\cdot 50t)$ и белый шум с σ = 0.05.
3. Попробуйте найти R-зубцы в зашумлённом сигнале напрямую (`signal.find_peaks` с порогом по высоте и минимальным расстоянием 0.25 с).
4. Выполните разложение `wavedec(y, 'db4', level=7)`. Составьте таблицу частотных полос уровней.
5. **Удаление дрейфа:** обнулите аппроксимацию и восстановите сигнал. Сравните с исходной ЭКГ.
6. **Поиск R-зубцов:** восстановите сигнал только по деталям уровней, в полосы которых попадает энергия QRS-комплекса (примерно 5–45 Гц), и найдите пики квадрата этого сигнала. Вычислите частоту сердечных сокращений.

In [ ]:
ecg = pywt.data.ecg().astype(float)
fs = 360
t = np.arange(len(ecg)) / fs
np.random.seed(4)
# Ваш код здесь

**Вопросы:**
- Почему прямой поиск пиков в зашумлённом сигнале не работает?
- Какие уровни разложения вы использовали для поиска R-зубцов и почему? Какова частота сердечных сокращений?

{Введите ваш ответ здесь}